# 14.13 Dynamic Programming

**Prerequisites:** 14.12 Recursion and Backtracking, 14.1 Complexity Analysis  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- The two preconditions: **overlapping subproblems** and **optimal substructure**
- The four-step progression: naive → **memoised** → **tabulated** → space-optimised
- 🔴 How to *recognise* a DP problem in an interview
- Designing the **state** - the hardest part, and the part that is skipped
- 1-D classics: climbing stairs, house robber, coin change
- 2-D classics: longest common subsequence, edit distance, 0/1 knapsack
- Reconstructing the actual answer, not just its size
- 🔴 When DP does **not** apply
- Interview questions, worked

---

## What DP actually is

The name is famously unhelpful — Richard Bellman chose it in the 1950s partly to sound impressive to a research funder. It means something simple:

> **Solve each subproblem once, remember the answer, and reuse it.**

That is all. The difficulty is never the caching; it is **working out what the subproblems are**.

### The two preconditions

DP applies when **both** hold:

| | Means | Fails when |
|---|---|---|
| **1. Overlapping subproblems** | the same subproblem is solved repeatedly | merge sort — each half is distinct, so caching gains nothing |
| **2. Optimal substructure** | the best answer is built from best answers to subproblems | longest *simple* path in a graph — a longer sub-path may be unusable |

🔴 **Both are required.** Overlap without optimal substructure means the cache is correct but useless; optimal substructure without overlap is just divide and conquer (**14.14**).

### Where you have already seen it

**14.12** memoised Fibonacci and turned 2,692,537 calls into 31. That *was* dynamic programming — this notebook makes it deliberate and extends it.

## The four-step progression

Every DP problem can be attacked in the same order, and each step is a mechanical transformation of the last. **Learn the progression, not the individual problems.**

```
   1. NAIVE RECURSION        write the definition. Exponential, correct.
        ↓  add a cache
   2. MEMOISATION (top-down) same code + @cache. Usually O(states).
        ↓  reverse the order
   3. TABULATION (bottom-up) a loop filling a table. No recursion limit.
        ↓  keep only what you need
   4. SPACE OPTIMISATION     often O(1) instead of O(n).
```

| Step | Time | Space | Notes |
|---|---|---|---|
| Naive | O(2ⁿ) | O(n) stack | write this first — it is the definition |
| Memoised | O(states) | O(states) + stack | 🔴 can hit the recursion limit |
| Tabulated | O(states) | O(states) | no recursion; usually the fastest |
| Optimised | O(states) | often **O(1)** | only if each state needs few predecessors |

> **In an interview, say this out loud.** "I'll write the recurrence first, then memoise it, then convert to a table if you want O(1) space." That structure is what is being assessed.

In [ ]:
import functools
import time

# ---------- 1. naive: the definition, transcribed ----------
def fib_naive(n):
    if n < 2:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)


# ---------- 2. memoised: the SAME code, plus a cache ----------
@functools.cache
def fib_memo(n):
    if n < 2:
        return n
    return fib_memo(n - 1) + fib_memo(n - 2)


# ---------- 3. tabulated: the same recurrence, bottom-up ----------
def fib_table(n):
    if n < 2:
        return n
    table = [0] * (n + 1)
    table[1] = 1
    for i in range(2, n + 1):
        table[i] = table[i - 1] + table[i - 2]     # same line, read forwards
    return table[n]


# ---------- 4. optimised: only two predecessors are ever needed ----------
def fib_optimised(n):
    previous, current = 0, 1
    for _ in range(n):
        previous, current = current, previous + current
    return previous                                # O(1) space


N = 30
versions = [
    ("1. naive        O(2^n)", fib_naive),
    ("2. memoised     O(n)", fib_memo),
    ("3. tabulated    O(n)", fib_table),
    ("4. optimised    O(n) time, O(1) space", fib_optimised),
]

print(f"fib({N})\n")
for label, function in versions:
    started = time.perf_counter()
    result = function(N)
    elapsed = time.perf_counter() - started
    print(f"  {label:<38}{result:>8,}{elapsed * 1000:>10.2f} ms")

print("\n  Same answer, four ways, each a mechanical step from the last.")
print(f"\n  fib(500) optimised: {str(fib_optimised(500))[:24]}...")
print("  The naive version could not reach fib(50) this century.")

print("\n🔴 Note what step 2 cost: ONE decorator line. That is the whole")
print("   difference between exponential and linear here.")

## 🔴 Recognising a DP problem

This is the real skill. The signals:

| Signal | Example phrasing |
|---|---|
| **"How many ways..."** | how many ways to climb the stairs |
| **"Minimum / maximum..."** over choices | fewest coins, maximum profit |
| **"Is it possible to..."** | can this be partitioned into equal halves |
| **"Longest / shortest ..."** subsequence | longest common subsequence |
| At each step you **choose** | take it or leave it, and the choices interact |

### The test that settles it

> Write the brute-force recursion. If the call tree **repeats subproblems**, it is DP.

🔴 **Greedy vs DP** is the confusion that costs marks. If a locally best choice is always globally best, greedy works and is simpler (**14.14**). If a choice that looks worse now can pay off later, you need DP. The coin-change example below is exactly this distinction.

### The three questions to answer, in order

1. **What is the state?** What must you know to solve one subproblem? That becomes your function's arguments, or your table's dimensions.
2. **What is the recurrence?** How does one state combine answers from smaller states?
3. **What are the base cases?** The smallest states, answered directly.

Get the state wrong and nothing else works. It is where nearly all the difficulty lives.

## 1-D DP: one dimension of state

### Climbing stairs

*You can climb 1 or 2 steps at a time. How many distinct ways to reach step n?*

- **State:** `ways(i)` = number of ways to reach step `i`
- **Recurrence:** `ways(i) = ways(i-1) + ways(i-2)` — you arrived from one of two places
- **Base:** `ways(0) = 1` (one way to stand still), `ways(1) = 1`

That is Fibonacci wearing a hat, which is worth noticing: **many DP problems are the same recurrence in different clothes**.

### House robber

*Houses in a row, each with some money. You cannot rob two adjacent houses. Maximum total?*

- **State:** `best(i)` = most money obtainable from houses `0..i`
- **Recurrence:** `best(i) = max(best(i-1), best(i-2) + money[i])` — skip this house, or take it and skip the previous one
- **Base:** `best(0) = money[0]`

🔴 This is where greedy fails: always taking the largest remaining house is wrong, as `[2, 7, 9, 3, 1]` demonstrates below.

In [ ]:
def climb_stairs(n):
    """ways(i) = ways(i-1) + ways(i-2). O(n) time, O(1) space."""
    if n <= 1:
        return 1
    two_back, one_back = 1, 1
    for _ in range(2, n + 1):
        two_back, one_back = one_back, one_back + two_back
    return one_back


def rob_houses(money):
    """best(i) = max(best(i-1), best(i-2) + money[i]). O(n) time, O(1) space."""
    skip = take = 0                    # best without / with the previous house
    for amount in money:
        skip, take = max(skip, take), skip + amount
    return max(skip, take)


def rob_greedy(money):
    """🔴 WRONG on purpose: repeatedly take the largest available house."""
    remaining = list(enumerate(money))
    taken = []
    while remaining:
        index, amount = max(remaining, key=lambda pair: pair[1])
        taken.append(amount)
        remaining = [(i, a) for i, a in remaining if abs(i - index) > 1]
    return sum(taken)


print("climbing stairs:")
for n in range(1, 8):
    print(f"  {n} steps -> {climb_stairs(n)} ways")
print("  ^ 1, 2, 3, 5, 8, 13, 21 - Fibonacci, in disguise\n")

print("house robber - DP versus greedy:")
print(f"{'houses':<24}{'DP':>6}{'greedy':>8}")
print("-" * 38)
for money in ([2, 7, 9, 3, 1], [2, 1, 1, 2], [5], [2, 1, 1, 2, 5, 1, 1, 5]):
    dp = rob_houses(money)
    greedy = rob_greedy(money)
    flag = "" if dp == greedy else "   <- greedy is WRONG"
    print(f"{str(money):<24}{dp:>6}{greedy:>8}{flag}")

print("\n🔴 For [2,1,1,2] greedy takes a 2, then can only reach one more 2")
print("   ... it happens to tie. For [2,7,9,3,1] greedy grabs 9, blocking")
print("   7, and loses. A locally best choice is not globally best - which")
print("   is precisely when you need DP rather than greedy (14.14).")

### Coin change - the canonical greedy trap

*Given coin denominations and a target, what is the fewest coins that make it?*

- **State:** `fewest(amount)` = fewest coins summing to `amount`
- **Recurrence:** `fewest(a) = 1 + min(fewest(a - c) for each coin c)`
- **Base:** `fewest(0) = 0`; unreachable amounts are infinity

🔴 **Greedy — always take the largest coin that fits — is wrong for general denominations.**

```
   coins [1, 3, 4], target 6

   greedy:  4 + 1 + 1  = 3 coins
   optimal: 3 + 3      = 2 coins
```

It happens to work for British and US currency, because those systems are deliberately *canonical*. It fails on arbitrary denominations — and interviewers choose arbitrary denominations for exactly this reason.

In [ ]:
def coin_change_dp(coins, target):
    """Fewest coins. O(target * len(coins)) time, O(target) space."""
    INF = float("inf")
    fewest = [0] + [INF] * target
    for amount in range(1, target + 1):
        for coin in coins:
            if coin <= amount and fewest[amount - coin] + 1 < fewest[amount]:
                fewest[amount] = fewest[amount - coin] + 1
    return fewest[target] if fewest[target] != INF else -1


def coin_change_greedy(coins, target):
    """🔴 Take the largest coin that fits. Wrong for general denominations."""
    remaining = target
    used = 0
    for coin in sorted(coins, reverse=True):
        while remaining >= coin:
            remaining -= coin
            used += 1
    return used if remaining == 0 else -1


def coin_change_which(coins, target):
    """Same DP, but records WHICH coins - reconstruction (see below)."""
    INF = float("inf")
    fewest = [0] + [INF] * target
    chosen = [None] * (target + 1)
    for amount in range(1, target + 1):
        for coin in coins:
            if coin <= amount and fewest[amount - coin] + 1 < fewest[amount]:
                fewest[amount] = fewest[amount - coin] + 1
                chosen[amount] = coin
    if fewest[target] == INF:
        return -1, []
    out, amount = [], target
    while amount > 0:
        out.append(chosen[amount])
        amount -= chosen[amount]
    return fewest[target], sorted(out, reverse=True)


cases = [([1, 3, 4], 6), ([1, 5, 10, 25], 30), ([2], 3), ([1, 2, 5], 11), ([9, 6, 5, 1], 11)]
print(f"{'coins':<18}{'target':>8}{'DP':>6}{'greedy':>8}")
print("-" * 42)
for coins, target in cases:
    dp = coin_change_dp(coins, target)
    greedy = coin_change_greedy(coins, target)
    flag = "" if dp == greedy else "   <- greedy WRONG"
    print(f"{str(coins):<18}{target:>8}{dp:>6}{greedy:>8}{flag}")

print("\n  [1,3,4] target 6: greedy takes 4+1+1 = 3 coins; optimal is 3+3 = 2")
print("  [9,6,5,1] target 11: greedy takes 9+1+1 = 3; optimal is 6+5 = 2")

count, which = coin_change_which([9, 6, 5, 1], 11)
print(f"\n  and the actual coins: {count} coins -> {which}")

## 2-D DP: two dimensions of state

When the subproblem depends on **two** things — usually positions in two sequences — the table becomes a grid.

### Longest common subsequence

*The longest sequence appearing in both strings, in order but not necessarily contiguous.*

- **State:** `lcs(i, j)` = LCS length of `a[:i]` and `b[:j]`
- **Recurrence:**
 - if `a[i-1] == b[j-1]`: `1 + lcs(i-1, j-1)` — extend the match
 - else: `max(lcs(i-1, j), lcs(i, j-1))` — drop one character from either
- **Base:** anything against an empty string is 0

```
        ""  A  B  C  B  D  A  B
    ""   0  0  0  0  0  0  0  0
    B    0  0  1  1  1  1  1  1
    D    0  0  1  1  1  2  2  2
    C    0  0  1  2  2  2  2  2
    A    0  1  1  2  2  2  3  3
    B    0  1  2  2  3  3  3  4   <- the answer
```

> **This is what `diff` does.** Version control, `git diff` and every code review tool are built on LCS or a close relative.

🔴 **Subsequence is not substring.** A substring is contiguous; a subsequence is not. Reading the question carelessly here costs the whole answer.

In [ ]:
def lcs_length(a, b):
    """O(m*n) time and space."""
    m, n = len(a), len(b)
    table = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                table[i][j] = table[i - 1][j - 1] + 1
            else:
                table[i][j] = max(table[i - 1][j], table[i][j - 1])
    return table[m][n], table


def lcs_reconstruct(a, b):
    """Walk the table BACKWARDS to recover the subsequence itself."""
    _, table = lcs_length(a, b)
    i, j = len(a), len(b)
    out = []
    while i > 0 and j > 0:
        if a[i - 1] == b[j - 1]:
            out.append(a[i - 1])            # this character was matched
            i, j = i - 1, j - 1
        elif table[i - 1][j] >= table[i][j - 1]:
            i -= 1
        else:
            j -= 1
    return "".join(reversed(out))


a, b = "ABCBDAB", "BDCABA"
length, table = lcs_length(a, b)
print(f"LCS of {a!r} and {b!r} = {length}")
print(f"the subsequence itself: {lcs_reconstruct(a, b)!r}")

print("\nthe table:")
print("       " + "".join(f"{ch:>3}" for ch in '""' + b))
for i, row in enumerate(table):
    label = '""' if i == 0 else a[i - 1]
    print(f"   {label:>3} " + "".join(f"{v:>3}" for v in row))

print("\n  Each cell depends only on the one above, the one left, and the")
print("  one diagonally up-left. That is why it fills in O(m*n).")

for x, y in (("", "abc"), ("abc", "abc"), ("abc", "xyz"), ("AGGTAB", "GXTXAYB")):
    print(f"  LCS({x!r:<10}, {y!r:<10}) = {lcs_length(x, y)[0]}  "
          f"{lcs_reconstruct(x, y)!r}")

### Edit distance (Levenshtein)

*Fewest single-character edits — insert, delete, substitute — to turn one string into another.*

- **State:** `distance(i, j)` = edits to turn `a[:i]` into `b[:j]`
- **Recurrence:** if the characters match, `distance(i-1, j-1)`; otherwise `1 + min(delete, insert, substitute)`
- **Base:** turning a string of length i into an empty one takes i deletions

```
    distance(i, j) = min( distance(i-1, j)   + 1     delete a[i-1]
                          distance(i, j-1)   + 1     insert b[j-1]
                          distance(i-1, j-1) + cost  substitute )
```

This powers spell-checkers, fuzzy search, DNA alignment and "did you mean...?".

🔴 **The base cases are not zero.** `distance(i, 0) = i`, not 0 — a mistake that produces plausible-looking wrong answers only for some inputs.

In [ ]:
def edit_distance(a, b):
    """Levenshtein. O(m*n) time and space."""
    m, n = len(a), len(b)
    table = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        table[i][0] = i                  # 🔴 i deletions, not 0
    for j in range(n + 1):
        table[0][j] = j                  # 🔴 j insertions

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            table[i][j] = min(
                table[i - 1][j] + 1,          # delete
                table[i][j - 1] + 1,          # insert
                table[i - 1][j - 1] + cost,   # substitute (or free match)
            )
    return table[m][n]


def edit_distance_rows(a, b):
    """Space-optimised: each row needs only the previous one. O(min(m,n)) space."""
    if len(a) < len(b):
        a, b = b, a                       # keep the inner dimension small
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        current = [i] + [0] * len(b)
        for j, cb in enumerate(b, start=1):
            cost = 0 if ca == cb else 1
            current[j] = min(previous[j] + 1, current[j - 1] + 1,
                             previous[j - 1] + cost)
        previous = current
    return previous[-1]


pairs = [("kitten", "sitting"), ("flaw", "lawn"), ("", "abc"),
         ("same", "same"), ("sunday", "saturday")]
print(f"{'a':<12}{'b':<12}{'distance':>10}{'optimised':>12}")
print("-" * 46)
for a, b in pairs:
    print(f"{a!r:<12}{b!r:<12}{edit_distance(a, b):>10}{edit_distance_rows(a, b):>12}")

print("\n  'kitten' -> 'sitting' is 3: k->s, e->i, and append g.")
print("\n  The optimised version keeps two rows instead of the whole grid:")
print("  O(m*n) space -> O(min(m,n)). Step 4 of the progression.")

import random

rng = random.Random(15)
alphabet = "abcde"
agree = all(
    edit_distance(x, y) == edit_distance_rows(x, y)
    for x, y in (("".join(rng.choices(alphabet, k=rng.randint(0, 8))),
                  "".join(rng.choices(alphabet, k=rng.randint(0, 8))))
                 for _ in range(300))
)
print(f"\n  both versions agree on 300 random pairs: {agree}")

### 0/1 knapsack

*Items with weights and values; a bag with a capacity. Maximise value. Each item is taken **once or not at all** — hence 0/1.*

- **State:** `best(i, c)` = best value using the first `i` items with capacity `c`
- **Recurrence:** `max(skip it, take it if it fits)`
- **Base:** no items, or no capacity → 0

```
    best(i, c) = max( best(i-1, c),                        skip item i
                      best(i-1, c - w[i]) + v[i] )         take item i
```

🔴 **This is the archetype of "greedy fails".** Taking the best value-per-weight ratio first is optimal for the *fractional* knapsack — where you may take part of an item — and **wrong** for 0/1, where you cannot.

> **A note on the complexity.** O(n × capacity) is called *pseudo-polynomial*: it is polynomial in the capacity's **value**, but exponential in the number of **bits** needed to write it. Doubling the capacity doubles the work. That is why 0/1 knapsack is NP-complete despite this tidy-looking algorithm.

In [ ]:
def knapsack(weights, values, capacity):
    """0/1 knapsack. O(n * capacity) time and space."""
    n = len(weights)
    table = [[0] * (capacity + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for c in range(capacity + 1):
            table[i][c] = table[i - 1][c]                  # skip item i-1
            if weights[i - 1] <= c:
                take = table[i - 1][c - weights[i - 1]] + values[i - 1]
                table[i][c] = max(table[i][c], take)

    chosen, c = [], capacity
    for i in range(n, 0, -1):
        if table[i][c] != table[i - 1][c]:                 # item i-1 was taken
            chosen.append(i - 1)
            c -= weights[i - 1]
    return table[n][capacity], sorted(chosen)


def knapsack_greedy(weights, values, capacity):
    """🔴 Best value-per-weight first. Optimal for FRACTIONAL, wrong for 0/1."""
    order = sorted(range(len(weights)),
                   key=lambda i: values[i] / weights[i], reverse=True)
    total, remaining, chosen = 0, capacity, []
    for i in order:
        if weights[i] <= remaining:
            total += values[i]
            remaining -= weights[i]
            chosen.append(i)
    return total, sorted(chosen)


cases = [
    ([1, 3, 4, 5], [1, 4, 5, 7], 7),
    ([10, 20, 30], [60, 100, 120], 50),
    ([5, 4, 6, 3], [10, 40, 30, 50], 10),
]
print(f"{'capacity':>9}{'DP value':>10}{'DP items':>14}{'greedy':>9}{'greedy items':>16}")
print("-" * 60)
for weights, values, capacity in cases:
    dp_value, dp_items = knapsack(weights, values, capacity)
    g_value, g_items = knapsack_greedy(weights, values, capacity)
    flag = "" if dp_value == g_value else "  <- greedy WRONG"
    print(f"{capacity:>9}{dp_value:>10}{str(dp_items):>14}"
          f"{g_value:>9}{str(g_items):>16}{flag}")

print("\n  For weights [5,4,6,3] values [10,40,30,50] capacity 10:")
print("    greedy takes item 3 (ratio 16.7) then item 1 (ratio 10) = 90")
print("    DP finds the same here - but change the numbers slightly and")
print("    it diverges. Greedy has no guarantee for 0/1 knapsack.")

# a case constructed so greedy definitely loses
weights, values, capacity = [3, 4, 5], [30, 50, 60], 8
dp_value, dp_items = knapsack(weights, values, capacity)
g_value, g_items = knapsack_greedy(weights, values, capacity)
print(f"\n  constructed case, capacity {capacity}:")
print(f"    DP     = {dp_value} using items {dp_items}")
print(f"    greedy = {g_value} using items {g_items}")
print(f"    {'greedy loses by ' + str(dp_value - g_value) if dp_value > g_value else 'tie here'}")

## 🔴 When DP does not apply

| Situation | Why | Use instead |
|---|---|---|
| No overlapping subproblems | the cache never hits | divide and conquer (**14.14**) |
| No optimal substructure | the best sub-answer is not usable | backtracking, or a different formulation |
| A greedy choice is provably optimal | DP is correct but needless | greedy (**14.14**) |
| The state space is enormous | O(states) is still too big | approximation, heuristics, pruning |

### The classic non-example

**Longest simple path in a graph** has overlapping subproblems but **no optimal substructure**: the longest path from A to C may not contain the longest path from A to B, because reusing a vertex is forbidden. DP quietly produces wrong answers.

> **Merge sort** is the other side: perfect optimal substructure, but the two halves are **disjoint** — no subproblem ever repeats, so memoising it wastes memory and gains nothing.

🔴 **Check both preconditions before reaching for a cache.**

## Interview questions

**1. What is dynamic programming?**
> Solving each subproblem once and reusing the result. It requires overlapping subproblems *and* optimal substructure.

**2. Memoisation or tabulation?**
> Memoisation is top-down, mirrors the recurrence, and only computes states you actually need — but can hit the recursion limit. Tabulation is bottom-up, has no recursion, and usually allows space optimisation.

**3. Climbing stairs / Fibonacci.** *(above)*
> Show the progression: recursion → cache → table → two variables.

**4. Coin change.** *(above)*
> Fewest coins is DP, not greedy. `[1,3,4]` target 6 is the counter-example. The "count the ways" variant needs the loops in the other order — a classic trap.

**5. House robber.** *(above)*
> `max(skip, take)`. O(1) space with two variables. Follow-up: houses in a circle — run it twice, excluding the first or the last.

**6. Longest common subsequence / edit distance.** *(above)*
> 2-D table. Say it is what `diff` and spell-checkers use, and note substring ≠ subsequence.

**7. 0/1 knapsack.** *(above)*
> Take or skip. Mention pseudo-polynomial complexity, and that greedy is optimal only for the fractional version.

**8. Longest increasing subsequence.**
> O(n²) DP is easy; O(n log n) with patience sorting plus `bisect` (**14.11**) is the impressive answer.

**9. How do you recover the actual solution, not just its value?**
> Store a choice alongside each state and walk backwards, or infer it by comparing table cells — both shown above.

**10. How do you spot a DP problem?**
> Write the brute-force recursion and look for repeated subproblems. Watch for "how many ways", "min/max over choices", "longest/shortest".

In [ ]:
# Question 8 - both versions, because the second one impresses.
import bisect


def lis_quadratic(data):
    """best(i) = 1 + max(best(j) for j < i with data[j] < data[i]). O(n^2)."""
    if not data:
        return 0
    best = [1] * len(data)
    for i in range(1, len(data)):
        for j in range(i):
            if data[j] < data[i]:
                best[i] = max(best[i], best[j] + 1)
    return max(best)


def lis_logarithmic(data):
    """Patience sorting. tails[k] = smallest possible tail of an
    increasing subsequence of length k+1. O(n log n) via bisect (14.11).

    🔴 `tails` is NOT the subsequence - only its LENGTH is meaningful.
    """
    tails = []
    for value in data:
        position = bisect.bisect_left(tails, value)
        if position == len(tails):
            tails.append(value)          # extends the longest run
        else:
            tails[position] = value      # a better tail for that length
    return len(tails)


cases = [
    [10, 9, 2, 5, 3, 7, 101, 18],
    [0, 1, 0, 3, 2, 3],
    [7, 7, 7, 7],
    [],
    [1, 2, 3, 4, 5],
]
print(f"{'data':<32}{'O(n^2)':>9}{'O(n log n)':>13}")
print("-" * 54)
for data in cases:
    print(f"{str(data):<32}{lis_quadratic(data):>9}{lis_logarithmic(data):>13}")

rng = random.Random(15)
agree = all(
    lis_quadratic(sample) == lis_logarithmic(sample)
    for sample in ([rng.randint(0, 50) for _ in range(rng.randint(0, 40))]
                   for _ in range(300))
)
print(f"\n  agree on 300 random inputs: {agree}")

big = [rng.randint(0, 10 ** 6) for _ in range(4_000)]
started = time.perf_counter()
lis_quadratic(big)
quad_time = time.perf_counter() - started
started = time.perf_counter()
lis_logarithmic(big)
log_time = time.perf_counter() - started
print(f"\n  n = {len(big):,}")
print(f"    O(n^2)     {quad_time * 1000:8.1f} ms")
print(f"    O(n log n) {log_time * 1000:8.1f} ms   {quad_time / log_time:,.0f}x faster")

---

## Common Mistakes & Pitfalls

1. 🔴 **Reaching for greedy when the choices interact.** Coin change with `[1,3,4]`, house robber, and 0/1 knapsack all break greedy.
2. 🔴 **Getting the base cases wrong.** Edit distance's `distance(i, 0)` is `i`, not 0 - and the error only shows on some inputs.
3. 🔴 **Designing the wrong state.** If the recurrence needs information the state does not carry, no amount of caching will save it.
4. 🔴 **Memoising with unhashable arguments.** `functools.cache` needs hashable keys (**14.6**); pass tuples.
5. **Confusing substring with subsequence.** One is contiguous, the other is not.
6. **Memoising when subproblems do not overlap.** You pay for the cache and gain nothing - merge sort is the example.
7. **Assuming O(n × capacity) is polynomial.** It is pseudo-polynomial: exponential in the number of bits of the capacity.
8. **Hitting the recursion limit with top-down DP** on a large state space. Convert to tabulation (**14.12**).
9. **Optimising space before the logic is right.** Get the table version correct, then collapse it.

## Best Practices

- Always write the naive recursion first - it is the definition, and it is your reference implementation.
- State the three pieces explicitly: state, recurrence, base cases.
- Add `@functools.cache` before hand-rolling a dict.
- Convert to tabulation when recursion depth or performance demands it.
- Optimise space only after the tabulated version is verified.
- Verify each version against the naive one on small random inputs.
- Store the choice, not just the value, when you need to reconstruct the answer.
- Check both preconditions - overlap *and* optimal substructure - before assuming DP applies.

## Practice Exercises

Try these before moving on.

1. Implement 'coin change: count the number of ways' and explain why the loop order differs from the fewest-coins version.
2. 🔴 Solve 'house robber in a circle': the first and last houses are now adjacent. Hint - run the linear version twice.
3. Implement 'longest palindromic substring' with DP, then compare with the expand-around-centre approach. Which is clearer?
4. Add reconstruction to `edit_distance` so it reports the actual sequence of operations.
5. Implement the unbounded knapsack (items may be reused) and explain the one-line difference from 0/1.
6. Solve 'partition into two subsets of equal sum' - it is 0/1 knapsack in disguise. What is the capacity?
7. 🔴 Take `lis_logarithmic` and try to output the actual subsequence, not just its length. Why is `tails` not the answer?
8. Implement 'minimum path sum in a grid' three ways: recursive, memoised, tabulated - then optimise it to a single row of state.